In [ ]:
from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole
import pandas as pd
import json
import re
import random
from pathlib import Path

In [ ]:
sber_key = ""

In [ ]:
def clean_json_response(response_text):
    cleaned = re.sub(r'```json|```', '', response_text).strip()
    cleaned = cleaned.strip()

    return cleaned

In [ ]:
SYSTEM_RULE = 'ВАЖНО: Не используй отсылки к структуре текста (абзацы, строки, "согласно источнику"). Вопрос должен звучать как самостоятельный поисковый запрос. Ответ должен быть краткой, но законченной фразой.'

prompt_1_classic = f'''{SYSTEM_RULE}
ЗАДАНИЕ: Напиши JSON по тексту.
УСЛОВИЯ:
1. Вопрос должен содержать факты (даты/города), чтобы быть понятным без текста.
2. Ответ ("gold_answer") должен быть коротким (НЕ БОЛЬШЕ 5 слов).
3. ЗАПРЕЩЕНО писать любой текст, кроме JSON.
ОТПРАВЬ МНЕ ТОЛЬКО JSON ВИДА: {{"question": "...", "gold_answer": "..."}}'''

# Промпт 2: Числа
prompt_2_numbers = f'''{SYSTEM_RULE}
ЗАДАНИЕ: Напиши JSON по тексту.
УСЛОВИЯ:
1. Найди в тексте число (дата, цена, количество) и укажи его с единицами измерения.
2. Вопрос должен подробно описывать ситуацию, чтобы число было единственно верным.
3. Ответ должен состоять из 2-3 слов.
4. ЗАПРЕЩЕНО писать любой текст, кроме JSON.
ОТПРАВЬ МНЕ ТОЛЬКО JSON ВИДА: {{"question": "...", "gold_answer": "..."}}'''

# Промпт 3: Ложная предпосылка (Ловушка)
prompt_3_false_premise = f'''{SYSTEM_RULE}
ЗАДАНИЕ: Напиши JSON по тексту.
УСЛОВИЯ:
1. Создай вопрос с ложным фактом (замени город/имя/дату из текста на похожую).
2. Ответ ("gold_answer") должен начинаться с "Нет, на самом деле...", а затем давать верный факт.
3. ЗАПРЕЩЕНО писать любой текст, кроме JSON.
ОТПРАВЬ МНЕ ТОЛЬКО JSON ВИДА: {{"question": "...", "gold_answer": "..."}}'''

# Промпт 4: Редкие термины и фамилии
prompt_4_rare = f'''{SYSTEM_RULE}
ЗАДАНИЕ: Напиши JSON по тексту.
УСЛОВИЯ:
1. Найди в тексте уникальную фамилию или термин.
2. Составь вопрос, описывающий роль этого объекта так детально, чтобы нельзя было подобрать синоним.
3. Ответ должен состоять из 1 слова.
4. ЗАПРЕЩЕНО писать любой текст, кроме JSON.
ОТПРАВЬ МНЕ ТОЛЬКО JSON ВИДА: {{"question": "...", "gold_answer": "..."}}'''

# Промпт 5: Причины
prompt_5_consequence = f'''{SYSTEM_RULE}
ЗАДАНИЕ: Напиши JSON по тексту.
УСЛОВИЯ:
1. Найди в тексте действие и его причину. Сформулируй вопрос "Зачем?" или "Почему?".
2. Ответ должен быть кратким (НЕ БОЛЬШЕ 5 слов), строго по фактам из источника.
3. ЗАПРЕЩЕНО писать любой текст, кроме JSON.
ОТПРАВЬ МНЕ ТОЛЬКО JSON ВИДА: {{"question": "...", "gold_answer": "..."}}'''

# Промпт 6: Определения
prompt_6_terms = f'''{SYSTEM_RULE}
ЗАДАНИЕ: Напиши JSON по тексту.
УСЛОВИЯ:
1. Выбери ключевой термин из текста.
2. Сформулируй вопрос: "Что именно понимается под термином [Имя] в контексте [Тема]?".
3. ЗАПРЕЩЕНО писать любой текст, кроме JSON.
ОТПРАВЬ МНЕ ТОЛЬКО JSON ВИДА: {{"question": "...", "gold_answer": "..."}}'''

# Промпт 7: Исключение
prompt_7_opposite = f'''{SYSTEM_RULE}
ЗАДАНИЕ: Напиши JSON по тексту.
УСЛОВИЯ:
1. Найди список из 3+ элементов. Сформулируй вопрос: "Какой из вариантов НЕ упоминается?".
2. В вопросе предложи 3 верных варианта из текста и 1 выдуманный (правдоподобный).
3. ЗАПРЕЩЕНО писать любой текст, кроме JSON.
ОТПРАВЬ МНЕ ТОЛЬКО JSON ВИДА: {{"question": "...", "gold_answer": "..."}}'''



In [ ]:
giga_lite = GigaChat(
    credentials=sber_key_1,
    verify_ssl_certs=False,
)

giga_pro = GigaChat(
    credentials=sber_key_1,
    verify_ssl_certs=False,
    model="GigaChat-Pro"
)

giga_max = GigaChat(
    credentials=sber_key_1,
    verify_ssl_certs=False,
    model="GigaChat-Max"
)

def ask_gigachat(prompt, question, giga_model):
    response = giga_model.chat(
        Chat(
            messages=[
                Messages(role=MessagesRole.SYSTEM, content=prompt[:500]),
                Messages(role=MessagesRole.USER, content=question[:500])
            ],
            temperature=1.0,
        )
    )

    return response

In [ ]:
dataset = []

In [ ]:
all_prompts = [
    prompt_1_classic,
    prompt_2_numbers,
    prompt_3_false_premise,
    prompt_4_rare,
    prompt_5_consequence,
    prompt_6_terms,
    prompt_7_opposite
]

THEME_CONFIG = {
    "география": {
        "prompts": [prompt_1_classic, prompt_2_numbers, prompt_7_opposite],
        "weights": [0.6, 0.3, 0.1]  # Приоритет классике и цифрам
    },
    "история": {
        "prompts": [prompt_1_classic, prompt_2_numbers, prompt_3_false_premise, prompt_5_consequence],
        "weights": [0.5, 0.2, 0.15, 0.15]
    },
    "науки": {
        "prompts": [prompt_1_classic, prompt_2_numbers, prompt_6_terms, prompt_4_rare],
        "weights": [0.4, 0.3, 0.2, 0.1] # В науке важны термины и цифры
    },
    "культура": {
        "prompts": [prompt_1_classic, prompt_4_rare, prompt_3_false_premise],
        "weights": [0.6, 0.3, 0.1]
    },
    "игры": {
        "prompts": [prompt_1_classic, prompt_4_rare, prompt_7_opposite],
        "weights": [0.5, 0.3, 0.2]
    },
    "общество": {
        "prompts": [prompt_1_classic, prompt_5_consequence, prompt_6_terms],
        "weights": [0.4, 0.3, 0.3] # Больше упора на причины и определения
    },
    "спорт": {
        "prompts": [prompt_1_classic, prompt_2_numbers, prompt_7_opposite],
        "weights": [0.5, 0.4, 0.1]
    }
}

WIKI_PARAMETER = 3
GIGA_MODEL = giga_lite


directory = Path("path_path")
theme_key = 'культура'
count = 0

current_prompts = THEME_CONFIG[theme_key]["prompts"]
current_weights = THEME_CONFIG[theme_key]["weights"]

for file_path in directory.glob('*.txt'):
    count += 1
    print(f"\n[{count}] Файл: {file_path.name}")
    try:
        content_context = file_path.read_text(encoding='utf-8')[:1500]
        previous_question = ""

        for i in range(1, WIKI_PARAMETER):
            chosen_prompt = random.choices(current_prompts, weights=current_weights, k=1)[0]

            extra_instruction = ""
            if i == 2 and previous_question:
                extra_instruction = f"\nВАЖНО: Твой первый вопрос был: '{previous_question}'. Сформулируй второй вопрос по ДРУГОЙ части текста, чтобы они не повторялись."

            full_task = chosen_prompt + extra_instruction

            print(f"   Запрос №{i} (Стратегия {all_prompts.index(chosen_prompt)})...")

            raw_response = ask_gigachat(full_task, content_context+'''ОТПРАВЬ МНЕ ТОЛЬКО JSON ВИДА: {{"question": "...", "gold_answer": "..."}}''', GIGA_MODEL).choices[0].message.content
            print(raw_response)
            clean_res = clean_json_response(raw_response)
            dict_answer = json.loads(clean_res)
            previous_question = dict_answer.get('question', "")

            row = {
                'file_name': file_path.name,
                'prompt': dict_answer.get('question'),
                'model_answer': None,
                'is_hallucination': None,
                'correct_answer': dict_answer.get('gold_answer'),
                'context': content_context,
                'strategy_used': all_prompts.index(chosen_prompt)
            }

            dataset.append(row)
            print(f"   Готово №{i}: {dict_answer.get('question')} {dict_answer.get('gold_answer')}")
    except Exception as e:
        print(f"   Ошибка на файле {file_path.name}: {e}")
        continue

In [ ]:
#Сохраняем датасет
df = pd.DataFrame(dataset)
df.to_csv('culture_non_wiki.csv', index=False, encoding='utf-16')